# 3-Model Ensemble: V5.0 + V6.0 + V7.0

In [1]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy numpy

import os, subprocess, zipfile, time, shutil, glob
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
print(f'Data: {len([d for d in os.listdir(DATA_DIR) if d.startswith("BraTS")])} cases')

# --- Patch fusion.py for bottleneck-only mode (V6.0 needs this) ---
with open('models/fusion.py', 'r') as f:
    code = f.read()
if 'bottleneck_only' not in code:
    old = '        return [\n'
    old += '            attn(feat, text_feat, text_mask)\n'
    old += '            for attn, feat in zip(self.attn_layers, features)\n'
    old += '        ]'
    new = '        if len(features) < len(self.attn_layers):\n'
    new += '            offset = len(self.attn_layers) - len(features)\n'
    new += '            return [\n'
    new += '                attn(feat, text_feat, text_mask)\n'
    new += '                for attn, feat in zip(self.attn_layers[offset:], features)\n'
    new += '            ]\n'
    new += '        return [\n'
    new += '            attn(feat, text_feat, text_mask)\n'
    new += '            for attn, feat in zip(self.attn_layers, features)\n'
    new += '        ]'
    code = code.replace(old, new)
    with open('models/fusion.py', 'w') as f:
        f.write(code)
    print('Patched fusion.py for bottleneck-only')

# --- Patch textmamba3d.py for fusion_mode ---
with open('models/textmamba3d.py', 'r') as f:
    code = f.read()
if 'fusion_mode' not in code:
    code = code.replace(
        "fusion_type: str = \"seqca\",",
        "fusion_type: str = \"seqca\",\n"
        "        fusion_mode: str = \"multi_scale\","
    )
    code = code.replace(
        "self.multi_scale_attn = fusion_cls(",
        "self.fusion_mode = fusion_mode\n"
        "        self.multi_scale_attn = fusion_cls("
    )
    old_fuse = '''            fused = self.multi_scale_attn(
                img_features[1:], text_features, attention_mask
            )'''
    new_fuse = '''            if self.fusion_mode == "bottleneck_only":
                shallow = img_features[1:-1]
                deep = [img_features[-1]]
                fused_deep = self.multi_scale_attn(deep, text_features, attention_mask)
                if self.text_gate is not None:
                    fused_deep = self.text_gate(deep, fused_deep)
                fused = list(shallow) + list(fused_deep)
            else:
                fused = self.multi_scale_attn(
                    img_features[1:], text_features, attention_mask
                )'''
    code = code.replace(old_fuse, new_fuse)
    with open('models/textmamba3d.py', 'w') as f:
        f.write(code)
    print('Patched textmamba3d.py for fusion_mode')

# --- Patch evaluate_full.py for fusion_mode ---
with open('evaluate_full.py', 'r') as f:
    code = f.read()
if 'fusion_mode' not in code:
    code = code.replace(
        "fusion_type=model_cfg.get('fusion_type', 'seqca'),",
        "fusion_type=model_cfg.get('fusion_type', 'seqca'),\n"
        "        fusion_mode=model_cfg.get('fusion_mode', 'multi_scale'),"
    )
    with open('evaluate_full.py', 'w') as f:
        f.write(code)
    print('Patched evaluate_full.py')

print('Setup complete')

Mounted at /content/drive
Mon Mar 30 23:00:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

In [2]:
# ===== Save V7.0 predictions =====
import os
os.chdir(REPO_DIR)

V70_CKPT = os.path.join(DRIVE_CKPT, 'best_V7.0.pth')
assert os.path.exists(V70_CKPT), f'V7.0 checkpoint not found: {V70_CKPT}'

PRED_DIR = os.path.join(DRIVE_BASE, 'ensemble_preds')
os.makedirs(f'{PRED_DIR}/v70', exist_ok=True)

print('Saving V7.0 predictions (text+TTA)...')
!python -u evaluate_full.py \
    --config configs/autoresearch/V7.0_boundary_hierarchy.yaml \
    --checkpoint "{V70_CKPT}" \
    --split test --overlap 0.5 \
    --use-text --tta \
    --save-preds "{PRED_DIR}/v70"

# Verify
saved = [f for f in os.listdir(f'{PRED_DIR}/v70') if f.endswith('_probs.npy')]
print(f'V7.0 predictions saved: {len(saved)} cases')

Saving V7.0 predictions (text+TTA)...
/content/TextMamba3D/models/__init__.py:1: UserWarning: Real Mamba3 not available, falling back to Mamba2.
  from .mamba_block import (
/content/TextMamba3D/models/mamba_block.py:357: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.dhw_fwd = _create_ssm(**ssm_kw)
/content/TextMamba3D/models/mamba_block.py:358: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.hwd_fwd = _create_ssm(**ssm_kw)
/content/TextMamba3D/models/mamba_block.py:359: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.wdh_fwd = _create_ssm(**ssm_kw)
config.json: 100% 385/385 [00:00<00:00, 1.93MB/s]
pytorch_model.bin: 100% 440M/440M [00:02<00:00, 168MB/s] 
Loading weights: 100% 199/199 [00:00<00:00, 1104.78it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: microso

## 3-Model Ensemble: Grid Search over Weights

In [3]:
# 3-model ensemble grid search (loop-inverted: load each case once)
import os, numpy as np
os.chdir(REPO_DIR)
import nibabel as nib

PRED_DIR = os.path.join(DRIVE_BASE, 'ensemble_preds')
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

v50_files = set(f for f in os.listdir(f'{PRED_DIR}/v50') if f.endswith('_probs.npy'))
v60_files = set(f for f in os.listdir(f'{PRED_DIR}/v60') if f.endswith('_probs.npy'))
v70_files = set(f for f in os.listdir(f'{PRED_DIR}/v70') if f.endswith('_probs.npy'))
common = sorted(v50_files & v60_files & v70_files)
print(f'Common cases: {len(common)}')
assert len(common) > 0, 'No common prediction files found!'

def dice_brats_regions(pred, gt):
    results = {}
    for name, classes in [('ET', [3]), ('TC', [1,3]), ('WT', [1,2,3])]:
        p = np.isin(pred, classes)
        g = np.isin(gt, classes)
        if p.sum() == 0 and g.sum() == 0:
            results[name] = 1.0
            continue
        intersection = (p & g).sum()
        results[name] = float(2 * intersection) / float(p.sum() + g.sum() + 1e-8)
    return results

# Weight combos
weight_combos = []
for w70 in [0.1, 0.15, 0.2, 0.25, 0.3]:
    for w50 in [0.3, 0.4, 0.5, 0.6, 0.7]:
        w60 = round(1.0 - w50 - w70, 2)
        if w60 >= 0.05:
            weight_combos.append((w50, w60, w70))
print(f'Testing {len(weight_combos)} weight combinations')

# Accumulate per-combo scores: combo_scores[idx] = {'ET': [], 'TC': [], 'WT': []}
combo_scores = [{k: [] for k in ['ET','TC','WT']} for _ in weight_combos]

# OUTER LOOP: cases (load each file ONCE)
for ci, fname in enumerate(common):
    case = fname.replace('_probs.npy', '')
    p50 = np.load(os.path.join(PRED_DIR, 'v50', fname))
    p60 = np.load(os.path.join(PRED_DIR, 'v60', fname))
    p70 = np.load(os.path.join(PRED_DIR, 'v70', fname))
    seg_path = os.path.join(DATA_DIR, case, f'{case}_seg.nii')
    if not os.path.exists(seg_path):
        seg_path += '.gz'
    gt = nib.load(seg_path).get_fdata().astype(np.int64)
    gt[gt == 4] = 3
    # INNER LOOP: all weight combos on this case
    for wi, (w50, w60, w70) in enumerate(weight_combos):
        ens = w50 * p50 + w60 * p60 + w70 * p70
        pred = np.argmax(ens, axis=0)
        d = dice_brats_regions(pred, gt)
        for k in ['ET','TC','WT']:
            combo_scores[wi][k].append(d[k])
    del p50, p60, p70, gt
    if (ci+1) % 20 == 0:
        print(f'  {ci+1}/{len(common)} cases processed')

# Compute means and find best
all_results = []
for wi, (w50, w60, w70) in enumerate(weight_combos):
    et = np.mean(combo_scores[wi]['ET'])
    tc = np.mean(combo_scores[wi]['TC'])
    wt = np.mean(combo_scores[wi]['WT'])
    mean = (et + tc + wt) / 3
    all_results.append((w50, w60, w70, et, tc, wt, mean))

all_results.sort(key=lambda x: x[6], reverse=True)
best = all_results[0]
print(f'\nTop 10:')
print(f'{"w50":>5} {"w60":>5} {"w70":>5} | {"ET":>7} {"TC":>7} {"WT":>7} {"Mean":>7}')
print('-' * 55)
for w50, w60, w70, et, tc, wt, mean in all_results[:10]:
    print(f'{w50:>5.2f} {w60:>5.2f} {w70:>5.2f} | {et:>6.4f} {tc:>6.4f} {wt:>6.4f} {mean:>6.4f}')
print(f'\nBest: w50={best[0]:.2f}, w60={best[1]:.2f}, w70={best[2]:.2f}, Mean={best[6]:.4f}')
print(f'Baseline V5.0: 0.8479')
print(f'2-model (V5.0+V6.0): 0.8496')
print(f'Delta vs baseline: {best[6]-0.8479:+.4f}')

Common cases: 95
Testing 24 weight combinations
  20/95 cases processed
  40/95 cases processed
  60/95 cases processed
  80/95 cases processed

Top 10:
  w50   w60   w70 |      ET      TC      WT    Mean
-------------------------------------------------------
 0.40  0.30  0.30 | 0.7921 0.8590 0.9022 0.8511
 0.30  0.40  0.30 | 0.7913 0.8598 0.9022 0.8511
 0.40  0.35  0.25 | 0.7918 0.8592 0.9020 0.8510
 0.30  0.45  0.25 | 0.7910 0.8601 0.9019 0.8510
 0.40  0.40  0.20 | 0.7914 0.8595 0.9017 0.8509
 0.30  0.50  0.20 | 0.7904 0.8605 0.9016 0.8509
 0.50  0.25  0.25 | 0.7924 0.8583 0.9017 0.8508
 0.50  0.30  0.20 | 0.7923 0.8585 0.9015 0.8508
 0.40  0.45  0.15 | 0.7909 0.8599 0.9014 0.8507
 0.30  0.55  0.15 | 0.7901 0.8607 0.9013 0.8507

Best: w50=0.40, w60=0.30, w70=0.30, Mean=0.8511
Baseline V5.0: 0.8479
2-model (V5.0+V6.0): 0.8496
Delta vs baseline: +0.0032


## Results Comparison

In [4]:
# ===== Summary table =====
print('=' * 70)
print('RESULTS COMPARISON')
print('=' * 70)
print(f'{"Model":<35} {"ET":>7} {"TC":>7} {"WT":>7} {"Mean":>7}')
print('-' * 63)
print(f'{"V5.0 (single)":<35} {"0.7910":>7} {"0.8560":>7} {"0.8967":>7} {"0.8479":>7}')
print(f'{"V6.0 (single)":<35} {"0.7770":>7} {"0.8646":>7} {"0.8991":>7} {"0.8469":>7}')
print(f'{"V7.0 (single)":<35} {"0.7848":>7} {"0.8570":>7} {"0.9032":>7} {"0.8483":>7}')
print(f'{"V5.0+V6.0 (2-model, w=0.65/0.35)":<35} {"0.7909":>7} {"0.8581":>7} {"0.8998":>7} {"0.8496":>7}')
print(f'{"TextBraTS SOTA":<35} {"0.8330":>7} {"0.8280":>7} {"0.8990":>7} {"0.8530":>7}')
print('-' * 63)
print('3-model ensemble results shown above ^^')

RESULTS COMPARISON
Model                                    ET      TC      WT    Mean
---------------------------------------------------------------
V5.0 (single)                        0.7910  0.8560  0.8967  0.8479
V6.0 (single)                        0.7770  0.8646  0.8991  0.8469
V7.0 (single)                        0.7848  0.8570  0.9032  0.8483
V5.0+V6.0 (2-model, w=0.65/0.35)     0.7909  0.8581  0.8998  0.8496
TextBraTS SOTA                       0.8330  0.8280  0.8990  0.8530
---------------------------------------------------------------
3-model ensemble results shown above ^^
